# **HANDS - ON: WHATSAPP CLOUD API**

## ✅ FIXED VERSION

Esta versión corrige el flujo de WhatsApp Cloud API para que las respuestas libres se envíen **solo después de un mensaje real del usuario a través de WhatsApp/Meta**. Un POST simulado a `localhost` sirve para probar el parser del webhook, pero **no abre la ventana real de atención de WhatsApp**.

También se eliminaron tokens incrustados en el notebook, se mejoró la verificación del webhook, se agregaron diagnósticos de errores de Meta y protección básica contra webhooks duplicados.


Una vez vista la masterclass ***WhatsApp Cloud API***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Ollama** para correr Llama 3.1 de forma local dentro de Colab, y **ngrok** para exponer el servidor mientras se desarrolla.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/19F84m5PKCMKEOHxL7z3MUqN98HEhwLWS?usp=drive_link).

## **LOGICA DE NEGOCIO Y MODELO**

### **CONFIGURACIÓN DEL ENTORNO**

#### **COLAB SECRETS**

Para no exponer tu API key directamente en el código, Colab ofrece un panel de Secrets (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la API key para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:

# Leer credenciales desde Colab Secrets
from google.colab import userdata

def secret_requerido(nombre):
    try:
        valor = userdata.get(nombre)
    except Exception:
        valor = None
    if not valor:
        raise ValueError(
            f"Falta el secreto '{nombre}' en Colab. "
            "Abre el panel Secrets (icono de llave), créalo y habilita acceso al notebook."
        )
    return valor

# Debe coincidir EXACTAMENTE con el Verify Token configurado en Meta.
VERIFY_TOKEN = secret_requerido("WHATSAPP_VERIFY_TOKEN")

# Credenciales de WhatsApp Cloud API
WHATSAPP_TOKEN = secret_requerido("WHATSAPP_ACCESS_TOKEN")
PHONE_NUMBER_ID = secret_requerido("PHONE_NUMBER_ID")

# ngrok
NGROK_AUTHTOKEN = secret_requerido("NGROK_AUTHTOKEN")

# API usada por todo el notebook
WHATSAPP_API_VERSION = "v25.0"

print("Credenciales cargadas correctamente desde Colab Secrets.")


#### **INSTALACIÓN Y ARRANQUE DE OLLAMA**

Ollama no viene preinstalado en la máquina virtual de Google Colab, por lo que se debe instalar y arrancar en segundo plano al inicio de cada sesión.

In [ ]:
# Instalar Ollama
!apt-get update -qq
!apt-get install -y zstd # requiere zstd, que Colab no trae por defecto
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Iniciar el servidor de Ollama
import subprocess
import time

proceso_ollama = subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # dar tiempo a que el servidor levante

!pip install ollama --quiet
import ollama

!ollama pull llama3.1:8b
print("Ollama listo.")

### **CONSTRUCCIÓN DE LA LÓGICA DEL ASISTENTE**

Con Ollama corriendo, se arma la lógica del asistente: la estructura del servidor con sus registros en memoria, la normalización de números mexicanos (necesaria para que la API de envío acepte el número que entrega el webhook), y la función que genera la respuesta real con Llama 3.1.

In [ ]:

# Configurar el servidor del webhook
from fastapi import FastAPI, Request, BackgroundTasks, HTTPException
from fastapi.responses import PlainTextResponse
import requests
import ollama
import re
import time

app = FastAPI()

mensajes_recibidos = []
usuarios_conocidos = set()
mensajes_procesados = set()
respuestas_enviadas = []


In [ ]:

# Normalizar un número para WhatsApp Cloud API
def normalizar_numero(numero):
    """
    Deja solamente dígitos. El número debe venir con código de país,
    por ejemplo 12025550123.
    """
    numero = re.sub(r"\D", "", str(numero))

    # Compatibilidad opcional con el formato mexicano antiguo 521...
    if numero.startswith("521") and len(numero) == 13:
        numero = "52" + numero[3:]

    return numero


In [ ]:
# Definir función de generación de respuesta
def generar_respuesta(texto_usuario):
    prompt = (
        "Eres el asistente de atención a clientes de un gimnasio. "
        "Responde de forma breve (2-3 líneas) a la siguiente pregunta sobre horarios o membresías:\n\n"
        + texto_usuario
    )

    respuesta = ollama.chat(model="llama3.1:8b", messages=[{"role": "user", "content": prompt}])
    return respuesta["message"]["content"]

## **DESPLIEGUE DEL SERVIDOR**

### **DEFINIR LOS ENDPOINTS DEL WEBHOOK**

Se definen los dos endpoints que WhatsApp necesita: uno para la verificación inicial del webhook, y otro para recibir cada mensaje entrante.

La recepción delega el trabajo pesado a `BackgroundTasks`, para responderle a Meta de inmediato sin esperar a que Llama genere la respuesta, evitando así que Meta reintente la entrega y se dupliquen las respuestas.

In [ ]:

# Endpoint de verificación de credenciales
@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params

    mode = params.get("hub.mode")
    token = params.get("hub.verify_token")
    challenge = params.get("hub.challenge")

    if mode == "subscribe" and token == VERIFY_TOKEN and challenge:
        # Meta espera que el body sea exactamente el challenge
        return PlainTextResponse(content=str(challenge), status_code=200)

    raise HTTPException(status_code=403, detail="Verify token inválido")


In [ ]:

# Extraer un mensaje de texto real desde un webhook de WhatsApp
def extraer_mensaje_texto(body):
    try:
        value = body["entry"][0]["changes"][0]["value"]

        # Los webhooks de estados (sent/delivered/read/failed) no traen "messages".
        mensajes = value.get("messages")
        if not mensajes:
            return None

        mensaje = mensajes[0]

        # Este ejercicio responde únicamente a texto.
        if mensaje.get("type") != "text":
            return None

        return {
            "id": mensaje.get("id"),
            "numero": normalizar_numero(mensaje["from"]),
            "texto": mensaje["text"]["body"],
        }
    except (KeyError, IndexError, TypeError):
        return None


# Endpoint de recepción de mensajes REALES desde Meta
@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)

    datos = extraer_mensaje_texto(body)

    if datos:
        mensaje_id = datos.get("id")

        # Meta puede reintentar un webhook. Evitamos responder dos veces.
        if mensaje_id and mensaje_id in mensajes_procesados:
            print("Webhook duplicado ignorado:", mensaje_id)
            return {"status": "duplicate_ignored"}

        if mensaje_id:
            mensajes_procesados.add(mensaje_id)

        print("Mensaje REAL recibido desde WhatsApp:", datos["numero"], "->", datos["texto"])
        background_tasks.add_task(
            procesar_y_responder,
            datos["numero"],
            datos["texto"]
        )

    return {"status": "ok"}


In [ ]:

# Enviar respuesta por WhatsApp Cloud API
def enviar_respuesta(numero, texto):
    numero = normalizar_numero(numero)

    url = (
        f"https://graph.facebook.com/"
        f"{WHATSAPP_API_VERSION}/{PHONE_NUMBER_ID}/messages"
    )

    headers = {
        "Authorization": f"Bearer {WHATSAPP_TOKEN}",
        "Content-Type": "application/json",
    }

    payload = {
        "messaging_product": "whatsapp",
        "recipient_type": "individual",
        "to": numero,
        "type": "text",
        "text": {
            "preview_url": False,
            "body": str(texto),
        },
    }

    try:
        r = requests.post(
            url,
            headers=headers,
            json=payload,
            timeout=30,
        )

        try:
            cuerpo = r.json()
        except Exception:
            cuerpo = r.text

        registro = {
            "numero": numero,
            "status_code": r.status_code,
            "response": cuerpo,
        }
        respuestas_enviadas.append(registro)

        print("WhatsApp API HTTP:", r.status_code)
        print("WhatsApp API respuesta:", cuerpo)

        if not r.ok:
            print(
                "\nERROR: Meta rechazó el envío. "
                "Revisa el objeto 'error' mostrado arriba. "
                "Si es un mensaje de texto libre, confirma que el usuario "
                "haya escrito REALMENTE al número de WhatsApp Business recientemente."
            )
            return None

        return cuerpo

    except requests.RequestException as e:
        print("ERROR DE RED enviando a WhatsApp:", repr(e))
        return None


In [ ]:

# Diagnóstico rápido del último envío
def mostrar_ultimo_envio():
    if not respuestas_enviadas:
        print("Todavía no hay intentos de envío registrados.")
        return

    ultimo = respuestas_enviadas[-1]
    print("Número:", ultimo["numero"])
    print("HTTP:", ultimo["status_code"])
    print("Respuesta de Meta:", ultimo["response"])


In [ ]:

# Procesar el mensaje y responder
def procesar_y_responder(numero, texto):
    try:
        if numero not in usuarios_conocidos:
            usuarios_conocidos.add(numero)
            enviar_respuesta(
                numero,
                "¡Hola! Soy el asistente del gimnasio. ¿En qué puedo ayudarte?"
            )
        else:
            respuesta = generar_respuesta(texto)
            enviar_respuesta(numero, respuesta)

    except Exception as e:
        print("ERROR procesando el mensaje:", repr(e))


### **EXPONER EL SERVIDOR CON NGROK**


Se levanta primero el servidor FastAPI y después se abre un túnel público con ngrok. La URL impresa terminada en `/webhook` debe configurarse como **Callback URL** en Meta for Developers. El **Verify Token** de Meta debe ser exactamente el mismo guardado en el secreto `WHATSAPP_VERIFY_TOKEN`.

> Nota: la URL gratuita de ngrok normalmente cambia cuando reinicias el runtime, por lo que debes actualizar la Callback URL en Meta cuando cambie.


\begin{equation} WhatsApp  \, \, \, \Rightarrow \, \, \, Configuración  \, \,de  \, \,la  \, \,API  \, \, \, \Rightarrow \, \, \, Configurar  \, \,webhooks \end{equation}

In [ ]:

# Instalar y arrancar servidor + túnel público con ngrok
!pip install pyngrok --quiet

from pyngrok import ngrok
import uvicorn
import threading
import time

# Cerrar túneles viejos de esta sesión, si existen
try:
    ngrok.kill()
except Exception:
    pass

# Si el servidor se ejecutó antes en este runtime, pedirle que termine
try:
    servidor.should_exit = True
    time.sleep(1)
except Exception:
    pass

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info",
)
servidor = uvicorn.Server(config)

def correr_servidor():
    servidor.run()

hilo = threading.Thread(target=correr_servidor, daemon=True)
hilo.start()

# Dar tiempo a Uvicorn para abrir el puerto
time.sleep(2)

ngrok.set_auth_token(NGROK_AUTHTOKEN)
tunel_publico = ngrok.connect(8000, "http")

CALLBACK_URL = tunel_publico.public_url + "/webhook"

print("Servidor iniciado.")
print("\nConfigura en Meta:")
print("Callback URL:", CALLBACK_URL)
print("Verify Token: usa el valor del secreto WHATSAPP_VERIFY_TOKEN")
print("\nDespués suscríbete al campo 'messages' del webhook.")


### **PROBAR EL FLUJO COMPLETO CORRECTAMENTE**



El flujo real debe probarse así:

1. Envía el template `hello_world` desde la API a tu teléfono de prueba.
2. **Desde tu teléfono, responde al mensaje dentro de WhatsApp.**
3. Esa respuesta viajará por Meta → ngrok → `/webhook`.
4. El bot podrá entonces responder con un mensaje de texto libre dentro de la conversación real.

⚠️ Un POST inventado a `http://127.0.0.1:8000/webhook` **no equivale** a recibir un mensaje de WhatsApp y no abre por sí solo la ventana real de conversación.


In [ ]:

# PRUEBA LOCAL DEL PARSER (NO envía WhatsApp y NO abre una conversación real)
payload_simulado = {
    "entry": [{
        "changes": [{
            "value": {
                "messages": [{
                    "id": "wamid.SIMULADO",
                    "from": "12025550123",
                    "type": "text",
                    "text": {"body": "Hola, esto es solo una prueba local"}
                }]
            }
        }]
    }]
}

print(extraer_mensaje_texto(payload_simulado))
print(
    "\nOK: el parser puede leer el payload. "
    "Esta celda NO llama al endpoint y NO intenta enviar una respuesta."
)


In [ ]:

# Enviar el template hello_world para iniciar la prueba desde Meta
# Guarda tu número de prueba (con código de país, solo dígitos) en:
# WHATSAPP_TEST_NUMBER  -> ejemplo: 12025550123

def enviar_template_hello_world(numero):
    numero = normalizar_numero(numero)

    url = (
        f"https://graph.facebook.com/"
        f"{WHATSAPP_API_VERSION}/{PHONE_NUMBER_ID}/messages"
    )

    headers = {
        "Authorization": f"Bearer {WHATSAPP_TOKEN}",
        "Content-Type": "application/json",
    }

    payload = {
        "messaging_product": "whatsapp",
        "recipient_type": "individual",
        "to": numero,
        "type": "template",
        "template": {
            "name": "hello_world",
            "language": {"code": "en_US"},
        },
    }

    r = requests.post(url, headers=headers, json=payload, timeout=30)

    try:
        respuesta = r.json()
    except Exception:
        respuesta = r.text

    print("HTTP:", r.status_code)
    print("Meta:", respuesta)

    if r.ok:
        print(
            "\n✅ Template aceptado por Meta. "
            "Ahora abre WhatsApp en ese teléfono y RESPONDE al mensaje."
        )
    else:
        print("\n❌ Meta rechazó el template. Revisa el objeto 'error' de arriba.")

    return respuesta


TEST_NUMBER = secret_requerido("WHATSAPP_TEST_NUMBER")
enviar_template_hello_world(TEST_NUMBER)


In [ ]:

# DESPUÉS DE RESPONDER DESDE TU WHATSAPP REAL, ejecuta esta celda para revisar actividad.
print("Payloads recibidos por /webhook:", len(mensajes_recibidos))
print("Intentos de respuesta por WhatsApp API:", len(respuestas_enviadas))

if mensajes_recibidos:
    print("\nÚltimo webhook recibido:")
    print(mensajes_recibidos[-1])

if respuestas_enviadas:
    print("\nÚltimo intento de envío:")
    mostrar_ultimo_envio()

print(
    "\nPara probar Llama: después de recibir el saludo del bot, "
    "manda un SEGUNDO mensaje desde tu WhatsApp real, por ejemplo: "
    "'¿A qué hora abren los sábados?'"
)


## **CHALLENGE: ASISTENTE DE PEDIDOS**

Una vez visto el ***Hands-On: Hands-On: WhatsApp Cloud API***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

A partir del asistente de pedidos construido, se agregan dos mejoras:

* Un mensaje de bienvenida para usuarios nuevos.
* Un tono de respuesta más controlado.

**IMPORTANTE:** Para su revisión, es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.

### **INSTRUCCIONES:**

**1. Prepara el entorno:** Instala y arranca Ollama en Colab (no viene preinstalado en la máquina virtual, así que se instala y se arranca en segundo plano al inicio de la sesión), descarga el modelo Llama 3.1 8B, y carga las credenciales de WhatsApp desde **Colab Secrets**.

In [ ]:
# Instalar Ollama


In [ ]:
# Iniciar el servidor de Ollama


In [ ]:
# Leer credenciales desde Colab Secrets


**2. Configura el servidor y el registro de usuarios conocidos:**

   * Crea el servidor FastAPI con las estructuras en memoria necesarias: `mensajes_recibidos` para la bitácora de payloads, y `usuarios_conocidos` (un `set`) para llevar registro de quién ya recibió el saludo.
   * Define la función `normalizar_numero_mx` para que los números mexicanos se envíen en el formato correcto.

In [ ]:
# Configurar el servidor del webhook


In [ ]:
# Definir función de normalización de número


**3. Genera la respuesta con un tono controlado:** Ajusta el prompt que se envía a Llama para que las respuestas se mantengan breves (máximo 2-3 líneas) y con un tono amable.

In [ ]:
# Definir función de generación de respuesta


**4. Define los endpoints del webhook:**

   * Crea el endpoint de verificación (`GET /webhook`).
   * Crea el endpoint de recepción de mensajes (`POST /webhook`), delegando el trabajo pesado a `BackgroundTasks` para responder de inmediato a Meta sin esperar a Ollama.

In [ ]:
# Definir endpoint de verificación de credenciales


In [ ]:
# Definir endpoint de recepción de mensajes


**5. Agrega el saludo inicial y el envío de la respuesta:**

   * Define la función que envía el mensaje de WhatsApp.
   * Define la función que decide si se manda el saludo fijo o la respuesta generada por Llama, según si el número ya está en `usuarios_conocidos`.

In [ ]:
# Definir función de envío de respuesta


In [ ]:
# Definir función de procesamiento de respuesta



**6. Expón el servidor y prueba el flujo completo:**

   * Instala `pyngrok`, arranca FastAPI y abre el túnel público.
   * Configura la URL pública `/webhook` en Meta y suscríbete al campo `messages`.
   * Envía un template aprobado (por ejemplo `hello_world`) al teléfono de prueba.
   * **Responde desde el WhatsApp real del usuario** para que Meta entregue un webhook real.
   * Confirma que el bot contesta; después manda un segundo mensaje real para probar la respuesta generada por Llama.
   * No uses un POST simulado a `localhost` como prueba end-to-end de WhatsApp; úsalo solo para probar el parser.


In [ ]:
# Installar e iniciar tunel publico con ngrok


In [ ]:
# Mandar mensaje al bot para recibir un saludo


In [ ]:
# Seguir conversando con el bot para interactuar con Llama
